# GStatSim on a GPU in Google Colab

Kriging and sequential Gaussian simulation (SGS) of Greenland bed topography with
[GStatSim](https://github.com/GatorGlaciology/GStatSim), run on a free Colab GPU with
[gstatsim-accel](https://github.com/alexislanier0420-tech/gstatsim-gpu). Nothing to install on your own computer.

**Before you start:** *Runtime → Change runtime type → T4 GPU → Save.*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alexislanier0420-tech/gstatsim-gpu/blob/main/notebooks/colab_quickstart.ipynb)

The data and steps follow the GStatSim book's chapters 3 (kriging) and 4 (SGS).
Every function takes `backend="gpu"` or `backend="cpu"`; the CPU backend runs GStatSim itself.

## 1. Install

In [ ]:
# Colab already has CuPy for its GPU; this adds GStatSim and gstatsim-accel.
%pip install -q "gstatsim-accel[bench] @ git+https://github.com/alexislanier0420-tech/gstatsim-gpu.git"

In [ ]:
import time

import cupy as cp
import gstatsim as gs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import skgstat as skg
from sklearn.preprocessing import QuantileTransformer

from gstatsim_accel import kriging, simulation

print("GPU:", cp.cuda.runtime.getDeviceProperties(0)["name"].decode())

## 2. Data: radar bed picks, gridded to 1 km and normal-score transformed

In [ ]:
url = "https://raw.githubusercontent.com/GatorGlaciology/GStatSim/main/demos/data/greenland_test_data.csv"
df = pd.read_csv(url)
df = df[df["Bed"] <= 700]          # the book drops picks above 700 m as bad picks

res = 1000
df_grid, _, _, _ = gs.Gridding.grid_data(df, "X", "Y", "Bed", res)
df_grid = df_grid[df_grid["Z"].notna()].reset_index(drop=True)

nst = QuantileTransformer(n_quantiles=500, output_distribution="normal", random_state=0)
df_grid["NZ"] = nst.fit_transform(df_grid[["Z"]].to_numpy()).ravel()

grid_xy = gs.Gridding.prediction_grid(df_grid["X"].min(), df_grid["X"].max(),
                                      df_grid["Y"].min(), df_grid["Y"].max(), res)
ny, nx = len(np.unique(grid_xy[:, 1])), len(np.unique(grid_xy[:, 0]))
extent = [grid_xy[:, 0].min() / 1e3, grid_xy[:, 0].max() / 1e3, grid_xy[:, 1].min() / 1e3, grid_xy[:, 1].max() / 1e3]
print(f"{len(df_grid):,} gridded data points, {len(grid_xy):,} cells to estimate ({ny} x {nx})")

## 3. Variogram (exponential model, as in the book)

In [ ]:
V = skg.Variogram(df_grid[["X", "Y"]].to_numpy(), df_grid["NZ"].to_numpy(),
                  bin_func="even", n_lags=70, maxlag=50_000, normalize=False)
V.model = "exponential"
rng, sill, nugget = (float(p) for p in V.parameters)
# GStatSim format: [azimuth, nugget, major range, minor range, sill, model]
vario = [0.0, nugget, rng, rng, sill, "Exponential"]
print(f"range {rng / 1e3:.1f} km, sill {sill:.2f}, nugget {nugget:.3f}")

## 4. Kriging the whole grid on the GPU

The first GPU call includes a few seconds of start-up (CUDA context, compiling kernels).

In [ ]:
kw = dict(num_points=100, radius=50_000, zz="NZ")
kriging.predict(df_grid, grid_xy[:200], vario, backend="gpu", **kw)   # warm-up

t = time.perf_counter()
est, var = kriging.predict(df_grid, grid_xy, vario, method="simple", backend="gpu", **kw)
print(f"GPU: {len(grid_xy):,} cells in {time.perf_counter() - t:.2f} s")

In [ ]:
bed = nst.inverse_transform(est.reshape(-1, 1)).reshape(ny, nx)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
im = ax[0].imshow(bed, extent=extent, cmap="gist_earth", vmin=-400, vmax=600)
fig.colorbar(im, ax=ax[0], label="Bed elevation (m)")
ax[0].set_title("Simple kriging estimate")
im = ax[1].imshow(var.reshape(ny, nx), extent=extent, cmap="Blues")
fig.colorbar(im, ax=ax[1], label="Kriging variance (normal scores)")
ax[1].set_title("Kriging variance")
for a in ax:
    a.set_xlabel("X (km)"); a.set_ylabel("Y (km)")
plt.tight_layout()

## 5. Same answers as GStatSim?

GStatSim on the CPU is slow, so check 500 cells. Most agree to ~1e-12. A few can differ
where two data points are exactly equally far away: GStatSim's pick between them is
arbitrary, so neither answer is wrong.

In [ ]:
sub = grid_xy[:500]
t = time.perf_counter()
est_cpu, _ = kriging.predict(df_grid, sub, vario, method="simple", backend="cpu", **kw)
t_cpu = time.perf_counter() - t
diff = np.abs(est_cpu - est[:500])
print(f"GStatSim: 500 cells in {t_cpu:.1f} s  (about {t_cpu / 500 * len(grid_xy) / 60:.0f} min for the whole grid)")
print(f"cells within 1e-6 of GStatSim: {np.mean(diff < 1e-6):.0%}   largest difference: {diff.max():.2g}")

## 6. Sequential Gaussian simulation: many realizations at once

SGS fills cells one after another, so one realization can't keep a GPU busy. Running many
realizations side by side can, and you need many anyway to map uncertainty.

In [ ]:
seeds = list(range(16))
t = time.perf_counter()
sims = simulation.realizations(df_grid, grid_xy, vario, seeds=seeds, backend="gpu", **kw)
dt = time.perf_counter() - t
print(f"{len(seeds)} realizations in {dt:.1f} s ({dt / len(seeds):.2f} s each)")

beds = nst.inverse_transform(sims.reshape(-1, 1)).reshape(len(seeds), ny, nx)
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for i in range(3):
    ax[i].imshow(beds[i], extent=extent, cmap="gist_earth", vmin=-400, vmax=600)
    ax[i].set_title(f"Realization {i + 1}")
im = ax[3].imshow(beds.std(axis=0), extent=extent, cmap="Blues")
fig.colorbar(im, ax=ax[3], label="m")
ax[3].set_title(f"Std. dev. of {len(seeds)} realizations")
for a in ax:
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout()

## Next steps

- The GPU versions of the book's chapters 3–9 are in
  [`notebooks/book/`](https://github.com/alexislanier0420-tech/gstatsim-gpu/tree/main/notebooks/book).
- Use your own data: any DataFrame with `X`, `Y` and a value column works the same way.
- On HiPerGator: see [`docs/hipergator.md`](https://github.com/alexislanier0420-tech/gstatsim-gpu/blob/main/docs/hipergator.md).